In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

%matplotlib qt
# %matplotlib notebook
import matplotlib.pyplot as plt

# %matplotlib notebook
from pathlib import Path
import numpy as np
import shutil
import sys
from typing import List, Dict, Tuple, Union, Set, Optional

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
import os
from pathlib import Path

from neuropy.io.neuroscopeio import NeuroscopeIO
from neuropy.io.binarysignalio import BinarysignalIO 
from neuropy.io.miniscopeio import MiniscopeIO

# from neuropy.core import Epoch, ProcessData
from neuropy.core.epoch import Epoch, EpochHelpers, ensure_dataframe, ensure_Epoch
from neuropy.io.spykingcircusio import SpykingCircusIO
from neuropy.core.session.data_session_loader import DataSessionLoader
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin, windows_to_wsl_path_if_needed, find_first_file_rglob


# Complete Test for Day1OpenField

In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin

## Bapun Format:
basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# basedir = Path('/Volumes/iNeo/Data/Bapun/Day5TwoNovel') # MacOS

# n_channels: int = 200
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day1Openfield'
excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


xml_path: Path = find_first_file_rglob(basedir, '*.xml', recursive=False)
xml_path

In [ ]:
sess = RawDataInitializationMixin.run_all(basedir=basedir,
                                        #   basename=None, excluded_data_datetimes=excluded_data_datetimes, n_channels=None, dat_file_sampling_rate=None,
                                          basename=basename, excluded_data_datetimes=excluded_data_datetimes, n_channels=n_channels, dat_file_sampling_rate=dat_file_sampling_rate, 
                                        #   valid_reference_session_basepath=windows_to_wsl_path_if_needed(Path("W:/Data/Bapun/RatS/Day5TwoNovel")), ref_basename = 'RatS-Day5TwoNovel-2020-12-04_07-55-09', ## reference session from which to create a new session (or copy missing probe/params/etc as needed)
)
sess

In [ ]:
## Determine `.paradigm.npy` (epochs) by looking at the position data
pos_df: pd.DataFrame = sess.position.to_dataframe().position.compute_speed_info()
pos_df

In [ ]:
sess.position.compute_speed_info()

In [ ]:
from neuropy.plotting.position import plot_position

ax = plot_position(position=sess.position)
ax

In [ ]:
from neuropy.utils.matplotlib_helpers import draw_epoch_regions, plot_position_curves_figure

sess.position.compute_speed_info() ## make sure we have velocity
fig, out_axes_list = plot_position_curves_figure(sess.position)


In [ ]:
## Unless otherwise provided, try to get the latest common timestamp for all of the session's data fields for the t_start and the earliest for the t_stop
session_t_start: float = np.nanmax([sess.position.t_start, sess.neurons.t_start])
session_t_stop: float = np.nanmin([sess.position.t_stop, sess.neurons.t_stop])
# (session_t_start, session_t_stop)
# 0.0, 5511.2575

# art_by_hand = Epoch(pd.DataFrame({"start": [246*60 + 31.1], "stop": [247*60 + 32.766], "label": "by_hand"}))
epochs_df: pd.DataFrame = pd.DataFrame({'start':[session_t_start],'stop':[session_t_stop],'label':['maze']})
epochs_df['duration'] = epochs_df['stop'] - epochs_df['start']
epochs_obj: Epoch = Epoch(epochs_df)
epochs_obj


if (sess.paradigm is None): #  or (sess.paradigm != epochs_obj)
    sess.paradigm = epochs_obj

    if epochs_obj is not None:
        epochs_obj.filename = sess.filePrefix.with_suffix('.paradigm.npy')
        print(f'saving out to {epochs_obj.filename.as_posix()}...')
        epochs_obj.save()
        print(f'\tdone.')



In [ ]:
sess.paradigm.add_epochs(epochs)



### Manual Processing

In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin

## INPUTS: basedir
replace_dict = RawDataInitializationMixin.step_update_session_files_n_channels(basedir=basedir, session_stem=basename, n_channels=195, dry_run=False, debug_print=True)
print(replace_dict)

In [ ]:
# basedir = sess.basedir
print(f'n_channels: {n_channels}')
print(f'dat_file_sampling_rate: {dat_file_sampling_rate}')
print(f'eeg_sampling_rate: {eeg_sampling_rate}')

raw_data_path: Path = basedir.joinpath('Raw_data')
assert raw_data_path.exists(), f"raw_data_path: '{raw_data_path.as_posix()}' does not exist!"
print(f'raw_data_path: "{raw_data_path.as_posix()}"')

## make the "spyk-circ" output directory
spyk_circ_output_dir: Path = basedir.joinpath('spyk-circ')
spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P

# ## INPUTS: raw_data_path: Path # Path(r'W:/Data/Bapun/RatS/Day1Openfield/Raw_data').resolve()
# print(f'raw_data_path: "{raw_data_path.as_posix()}"')

# ## find all constitutent "continuous.dat" files recurrsively in all subdirectories: "W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_10-24-24\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat"
# found_raw_data_paths = ["W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-20-27/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         # "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-24-24/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat", ## BAD ONE, only has 32 channels, skip
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_13-02-47/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_14-30-32/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_15-06-02/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
# ]    
# ## *-24-24 is a bad one with only 30 good channels!

# ## Iterate through and make proper paths, check their existance
# found_raw_data_paths: List[Path] = [Path(v).resolve() for v in found_raw_data_paths]
## could assert that they all exist... but let's NOT!
# excluded_data_datetimes = ['2020-11-25_10-24-24']


included_only_datetime_df, datetime_csv_out_path, found_raw_data_paths, all_datetime_df, all_found_files_dict = RawDataInitializationMixin.build_session_datetime_csv(raw_data_path, basename=basename, excluded_data_datetimes=excluded_data_datetimes,
                                                                                                                n_channels=n_channels, sampling_rate=dat_file_sampling_rate,
                                                                                        )
included_only_datetime_df
all_datetime_df

In [ ]:
raw_data_derived_n_channels_list = included_only_datetime_df['n_channels'].tolist()
if len(raw_data_derived_n_channels_list) == 0:
    print(f'WARNING: raw_data_derived_n_channels_list is empty!')
    
raw_data_derived_n_channels: int = raw_data_derived_n_channels_list[0]
np.all(raw_data_derived_n_channels_list) == raw_data_derived_n_channels_list[0], f"raw_data_derived_n_channels_list entires must all be the same but: {raw_data_derived_n_channels_list}"


In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin

## INPUTS: basedir
replace_dict = RawDataInitializationMixin.step_update_session_files_n_channels(basedir=basedir, session_stem=basename, n_channels=195, dry_run=True, debug_print=True)
print(replace_dict)

In [ ]:
# dat_files = all_datetime_df['dat_file'].values.tolist()

dat_files = all_datetime_df[all_datetime_df['is_included']]['dat_file'].values.tolist()
dat_file_sizes: List[int] = [Path(p).stat().st_size for p in dat_files]
dat_files
dat_file_sizes

In [ ]:
sum_file_size_bytes: int = np.sum(dat_file_sizes)
sum_file_size_bytes

In [ ]:
concatenated_file_output_path: Path = spyk_circ_output_dir.joinpath(f'{basename}.dat').resolve()
concatenated_file_size_bytes: int = concatenated_file_output_path.stat().st_size
concatenated_file_size_bytes

In [ ]:
concatenated_file_size_bytes - sum_file_size_bytes

In [ ]:
found_raw_data_paths

In [ ]:
## Copy the concatenated files to the output directory
concatenated_file_output_path: Path = RawDataInitializationMixin.step_perform_concat(found_raw_data_paths=found_raw_data_paths, spyk_circ_output_dir=spyk_circ_output_dir, basename=basename, force_overwrite=False)
print(f'have concatenated_file_output_path: "{concatenated_file_output_path.as_posix()}"')
concatenated_file_output_path

In [ ]:

eval "$(micromamba shell hook --shell=bash)"
micromamba activate circus


## Normal Neuropy Bapun session loading

In [ ]:
## Bapun Format:
# basedir = '/media/share/data/Bapun/Day5TwoNovel' # Linux
basedir = Path('W:/Data/Bapun/RatS/Day1Openfield') # Windows
# basedir = '/Volumes/iNeo/Data/Bapun/Day5TwoNovel' # MacOS

raw_data_path: Path = basedir.joinpath('Raw_data')
print(f'raw_data_path: "{raw_data_path.as_posix()}"')
assert raw_data_path.exists()

## make the "spyk-circ" output directory
spyk_circ_output_dir: Path = basedir.joinpath('spyk-circ')
spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P

# ## INPUTS: raw_data_path: Path # Path(r'W:/Data/Bapun/RatS/Day1Openfield/Raw_data').resolve()
# print(f'raw_data_path: "{raw_data_path.as_posix()}"')

# ## find all constitutent "continuous.dat" files recurrsively in all subdirectories: "W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_10-24-24\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat"
# found_raw_data_paths = ["W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-20-27/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         # "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-24-24/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat", ## BAD ONE, only has 32 channels, skip
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_13-02-47/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_14-30-32/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_15-06-02/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
# ]    
# ## *-24-24 is a bad one with only 30 good channels!

# ## Iterate through and make proper paths, check their existance
# found_raw_data_paths: List[Path] = [Path(v).resolve() for v in found_raw_data_paths]
## could assert that they all exist... but let's NOT!
excluded_data_datetimes = ['2020-11-25_10-24-24']


In [ ]:
n_channels: int = 200
dat_file_sampling_rate: int = 30000
basename: str = 'RatS-Day1Openfield'
excluded_data_datetimes = ['2020-11-25_10-24-24']

datetime_df, datetime_csv_out_path, found_raw_data_paths = RawDataInitializationMixin.build_session_datetime_csv(raw_data_path, basename=basename, n_channels=n_channels, sampling_rate=dat_file_sampling_rate,
																												 excluded_data_datetimes=excluded_data_datetimes, minimum_recording_duration_hours=None)
datetime_df
# found_raw_data_paths

# for path, dt in results.items():
#     print(f"{path.parent.name} -> {dt.isoformat()}")

In [ ]:
## INPUTS: basedir
sess = DataSessionLoader.bapun_data_session(basedir, enable_continue_on_required_path_failure=True)
active_sess_config = sess.config
session_name = sess.name

active_sess_config
print(f'session_name: {session_name}')


In [ ]:
# sess.eegfile ## has an EEG file object
output_process_resample_cmd: Optional[str] = RawDataInitializationMixin.build_process_resample_command(sess, n_channels=n_channels, dat_sampling_rate_Hz=sampling_rate, desired_eeg_sampling_rate_Hz=1250)
output_process_resample_cmd

### Build the `.datetime.csv` information from the raw files used.

In [ ]:
## Copy the concatenated files to the output directory
concatenated_file_output_path: Path = RawDataInitializationMixin.step_perform_concat(found_raw_data_paths=found_raw_data_paths, spyk_circ_output_dir=spyk_circ_output_dir)
print(f'have concatenated_file_output_path: "{concatenated_file_output_path.as_posix()}"')
concatenated_file_output_path


In [ ]:
print(sess.recinfo)

In [ ]:
recinfo_dict = sess.recinfo.to_dict()
recinfo_dict

In [ ]:

print('session dataframe spikes: {}\n'.format(sess.spikes_df.shape))

print('session dataframe spikes: {}\nsession.neurons.n_spikes summed: {}\n'.format(sess.spikes_df.shape, np.sum(sess.neurons.n_spikes)))

# sess = sessions[0]

### Assign position data

In [ ]:
from neuropy.core import Position

position_csvs_path: Path = sess.basepath.joinpath('Raw_data/position/CSVs')
position: Position = RawDataInitializationMixin.build_initial_position_data(sess, position_csvs_path=position_csvs_path)
position

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

# plt.plot(opti_data.datetime_array,opti_data.z)
plt.plot(sess.position.x, sess.position.y)

### Linearize position

In [ ]:
from neuropy.utils import position_util

maze = sess.paradigm['maze']
maze_pos = sess.position.time_slice(maze[0],maze[1])
linear_pos = position_util.linearize_position(maze_pos)

In [ ]:
linear_pos.y

In [ ]:
%matplotlib widget
plt.plot(maze_pos.time,maze_pos.x)
plt.plot(linear_pos.time,linear_pos.x)

In [ ]:
import numpy as np

a = np.ones(5)
np.vstack((a,a,a)).shape

In [ ]:
from neuropy.core import animal

d = {'name':'hello','tag':'ser'}

an = animal.Animal(d) 


In [ ]:
an.name

# Neurons
- import spiketrains from Phy
- estimate neuron types: pyramidal, interneuron and mua. 

## Importing spiketrains from Phy

In [ ]:
def fnImportBapunData(data_load_path, basename: str='RatS-Day5TwoNovel-2020-12-04_07-55-09'):
    data_filenames = [f'{basename}.neurons.npy', f'{basename}.paradigm.npy', f'{basename}.position.npy', f'{basename}.probegroup.npy']
    # data_variable_names = [['spiketrains', 't_stop', 't_start', 'sampling_rate', 'neuron_ids', 'neuron_type', 'waveforms', 'peak_channels', 'shank_ids'], ['epochs'], ['traces', 't_start', 'sampling_rate'], ['data']]
    data_variable_names = [['spiketrains', 't_stop', 't_start', 'sampling_rate', 'neuron_ids', 'neuron_type', 'waveforms', 'peak_channels', 'shank_ids'], ['epochs'], ['df'], ['data']]
    ## Load Bapun's Saved Data:    

    all_loaded_data = {} # new dictionary

    for i in np.arange(len(data_filenames)):
        active_filename = data_load_path.joinpath(data_filenames[i])
        print('active_filename: {}'.format(active_filename))
        if active_filename.exists():
            #with open(active_filename, 'rb') as f:
            #with np.load(active_filename, allow_pickle=True) as loaded_data:
            loaded_data = np.load(active_filename, allow_pickle=True).item() # need .item() to recover the array item
            # print('loaded_data[0]: {}'.format(loaded_data))
            print(loaded_data)
            
            for j in np.arange(len(data_variable_names[i])):
                all_loaded_data[data_variable_names[i][j]] = loaded_data[data_variable_names[i][j]]

            print('\tloaded loaded_data from {}'.format(active_filename))
        else:
            print('\tdoes not exist. skipping.')
    ## end for i in ...
    return all_loaded_data, data_variable_names
    
all_loaded_data, data_variable_names = fnImportBapunData(data_load_path=sess.basepath, basename=sess.config.session_name)
all_loaded_data

## Get probegroup
# sess.probegroup

In [ ]:
from neuropy.core.probe import Shank, Probe, ProbeGroup

# probe_file = sess.filePrefix.with_suffix('.probegroup.npy')
probe_file = Path(r"W:\Data\Bapun\RatS\Day5TwoNovel\RatS-Day5TwoNovel-2020-12-04_07-55-09.probegroup.npy").resolve()
assert probe_file.exists(), f"probe_file: '{probe_file.as_posix()}'"



In [ ]:
probe_group: ProbeGroup = ProbeGroup.from_file(probe_file)
probe_group

In [ ]:
probe_group.get_channels(groupby='shank')

In [ ]:
from neuropy.io.neuroscopeio import NeuroscopeIO

# nrs_path = Path(r"W:\Data\Bapun\RatK\Day4Openfield\RatK_Day4_2019-08-16_04-42-36.nrs").resolve()
ref_nrs_path = Path(r"W:\Data\Bapun\RatS\Day5TwoNovel\RatS-Day5TwoNovel-2020-12-04_07-55-09.nrs").resolve()
ref_xml_path = Path(r"W:\Data\Bapun\RatS\Day5TwoNovel\RatS-Day5TwoNovel-2020-12-04_07-55-09.xml").resolve()

ref_neuroscope_obj: NeuroscopeIO = NeuroscopeIO(xml_filename=ref_xml_path)
ref_good_channels = ref_neuroscope_obj.good_channels
print(f'ref_good_channels: {ref_good_channels}')
ref_channel_groups_dict: Dict[int, np.array] = {(grp_idx+1):channels_list for grp_idx, channels_list in enumerate(ref_neuroscope_obj.channel_groups.tolist())} ## 1-indexed dict of channel groups
ref_channel_groups_dict


In [ ]:

neuroscope_dict = neuroscope_obj.to_dict()
neuroscope_dict

In [ ]:
xml_path = Path(r"W:\Data\Bapun\RatS\Day1Openfield\RatS-Day1Openfield.xml").resolve()
neuroscope_obj: NeuroscopeIO = NeuroscopeIO(xml_filename=xml_path)
good_channels = neuroscope_obj.good_channels
print(f'good_channels: {good_channels}')
channel_groups_dict: Dict[int, np.array] = {(grp_idx+1):channels_list for grp_idx, channels_list in enumerate(neuroscope_obj.channel_groups.tolist())} ## 1-indexed dict of channel groups
channel_groups_dict

In [ ]:
is_channel_missing_original = np.isin(ref_good_channels, good_channels)
assert np.all(is_channel_missing_original), f"the later reference session should not contain any channels that don't exist in the unfiltered session.\n\tref_good_channels (ref only): {ref_good_channels[is_channel_missing_original]}"
is_channel_marked_bad_ref = np.logical_not(np.isin(good_channels, ref_good_channels))
channels_marked_bad_in_ref = good_channels[is_channel_marked_bad_ref]
channels_marked_bad_in_ref

In [ ]:
## Update curr from ref
neuroscope_obj.skipped_channels = ref_neuroscope_obj.skipped_channels
neuroscope_obj.discarded_channels = ref_neuroscope_obj.discarded_channels
neuroscope_obj._good_channels() ## update good channels on the object
neuroscope_obj.channel_groups = ref_neuroscope_obj.channel_groups
_bak_xml_path = neuroscope_obj.backup_xml_file()
neuroscope_obj.update_xml_file()

In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin, windows_to_wsl_path_if_needed

win_path: Path = Path(r"W:/Data/Bapun/RatS/Day5TwoNovel/RatS-Day5TwoNovel-2020-12-04_07-55-09.xml")
win_path

In [ ]:
windows_to_wsl_path_if_needed(path=win_path)

In [ ]:
win_path_str: str = win_path.as_posix()
win_path_Str



In [ ]:

# wsl_root_suffix: str = f'/mnt/{}'

"W:/Data/Bapun/RatS/Day5TwoNovel"
"/mnt/w/Data/Bapun/RatS/Day5TwoNovel"


In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin, windows_to_wsl_path_if_needed

neuroscope_obj = RawDataInitializationMixin.step_deploy_spyking_circus_files_from_template(
                                    # valid_reference_session_basepath=Path("/mnt/w/Data/Bapun/RatS/Day5TwoNovel"), ref_basename = 'RatS-Day5TwoNovel-2020-12-04_07-55-09',
                                    valid_reference_session_basepath=windows_to_wsl_path_if_needed(Path("W:/Data/Bapun/RatS/Day5TwoNovel")), ref_basename = 'RatS-Day5TwoNovel-2020-12-04_07-55-09',
									target_session_basepath=sess.basepath, target_basename = sess.name)
neuroscope_obj

In [ ]:
from neuropy.io import PhyIO
from neuropy.core import Neurons
from pathlib import Path
import numpy as np

# cluster_path = Path(
#     # "/data/Clustering/SleepDeprivation/RatN/Day1/spykcirc/clus_combined"
#     windows_to_wsl_path_if_needed(r"W:/Data/Bapun/RatS/Day1Openfield/spyk-circ/RatS-Day1Openfield")
#     # windows_to_wsl_path_if_needed(r"W:/Data/Bapun/RatS/Day1Openfield/spyk-circ/RatS-Day1Openfield/RatS-Day1Openfield-merged.GUI")
# )
# assert cluster_path.exists(), f'cluster_path does not exist: "{cluster_path.as_posix()}"'
# chan_grps = sess.recinfo.channel_groups
# sub_dirs = sorted([x for x in cluster_path.iterdir() if x.is_dir()])

# spk, peak_chans, waveforms, shank_id = [], [],[],[]
# for f_ind, folder in enumerate(sub_dirs):
#     print(f'trying folder: "{folder.as_posix()}"...')
#     try:
#         phy_data = PhyIO(folder)
#         spk.extend(phy_data.spiketrains)
#         peak_chans.extend(chan_grps[f_ind][phy_data.peak_channels])
#         waveforms.extend(phy_data.peak_waveforms)
#         shank_id.extend([f_ind]*len(phy_data.spiketrains))
#         print(f'\tdone successfully with folder!')
#     except Exception as e:
#         print(f'\tfailed for folder with error {e}. Continuing...')
#         # raise e
    
# peak_chans = np.array(peak_chans)
# peak_chans

phy_path = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield/spyk-circ/RatS-Day1Openfield/RatS-Day1Openfield-merged.GUI")
assert phy_path.exists()

# spk, peak_chans, waveforms, shank_id = [], [],[],[]
phy_data: PhyIO = PhyIO(phy_path)

# spk.extend(phy_data.spiketrains)
# # peak_chans.extend(chan_grps[f_ind][phy_data.peak_channels])
# peak_chans.extend(phy_data.peak_channels)   # already global

# waveforms.extend(phy_data.peak_waveforms)
# shank_id.extend([f_ind]*len(phy_data.spiketrains))
# shank_id.extend([int(v) for v in phy_data.shank_ids])   # from cluster_info.tsv "sh" column
print(f'\tdone successfully with folder!')

neurons: Neurons = Neurons(
    np.array(phy_data.spiketrains, dtype=object),
    t_stop=sess.eegfile.duration,
    sampling_rate=phy_data.sampling_rate,
    peak_channels=phy_data.peak_channels,
    waveforms=np.array(phy_data.peak_waveforms,dtype='object'),
    shank_ids=np.array([int(v) for v in phy_data.shank_ids])
)

neurons

In [ ]:
if neurons is not None:
    sess.neurons = neurons

## Estimate neuron type

In [ ]:
# %matplotlib inline
# import matplotlib.pyplot as plt

from neuropy.utils import neurons_util
neuron_type = neurons_util.estimate_neuron_type(sess.neurons)
neuron_type = neuron_type[0]
neuron_type
neurons.neuron_type = neuron_type


In [ ]:
sess.neurons.filename = sess.filePrefix.with_suffix('.neurons')
sess.neurons.save()

In [ ]:
import matplotlib.pyplot as plt
from neuropy.plotting.spikes import plot_raster

plot_raster(neurons,color='jet',add_vert_jitter=True)

## BinnedSpiketrain and Mua objects using Neurons

In [ ]:
mua = sess.neurons.get_mua()
mua.filename = sess.filePrefix.with_suffix(".mua.npy")
if mua is not None:
    sess.mua = mua
    mua.save() 


In [ ]:
smth_mua = sess.mua.get_smoothed(sigma=0.02)

In [ ]:
from neuropy.plotting.spikes import plot_mua

plot_mua(smth_mua)

In [ ]:
from neuropy.analyses.spkepochs import detect_pbe_epochs

pbe = detect_pbe_epochs(smth_mua)
pbe.filename = sess.filePrefix.with_suffix('.pbe')
pbe.save()


## Identify disconnect and motion artifacts

In [ ]:
signal = sess.eegfile.get_signal()

In [ ]:
from neuropy.analyses.artifact import detect_artifact_epochs

art_epochs_file = sess.filePrefix.with_suffix('.artifact.npy')
art_epochs = Epoch.from_file(art_epochs_file)
if art_epochs is None:
    art_epochs = detect_artifact_epochs(signal, thresh=8, edge_cutoff=2, merge=6)
    sess.recinfo.write_epochs(epochs=art_epochs, ext='art')
    art_epochs.filename = art_epochs_file
    art_epochs.save()

art_epochs

### Sanity check plot

In [ ]:
from neuropy.plotting.epochs import plot_artifact_epochs

# _, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
# ax = axs[0]
plot_artifact_epochs(epochs=art_epochs, signal=signal) # , channel=27, ax=ax


### Manually fix / add in any missing artifacts identified above.

In [ ]:
# from copy import deepcopy

# # Manually fix missing noise epoch in middle!
# art_list = [[1316, 1410.5], [1800, 1805]]  # This code would add in two artifact epochs from 1316-1410.5 and 1800-1805.
# art_epochs_fix = deepcopy(art_epochs)
# for (start, stop) in art_list:
#     art_epochs_fix = art_epochs_fix.add_epoch_manually(start, stop)

# sess.recinfo.write_epochs(epochs=art_epochs_fix, ext='art')
# art_epochs_fix.save(art_epochs_source_file)

In [ ]:
# # Alternate code to add in missing artifacts, e.g. from loosening of headstage not caught 
# # well by automatic artifact detection
# art_by_hand = Epoch(pd.DataFrame({"start": [246*60 + 31.1], "stop": [247*60 + 32.766], "label": "by_hand"}))

# art_epochs_combined = pd.concat((art_epochs._epochs, art_by_hand._epochs), 
#                                 keys=["start", "stop", "label"], ignore_index=True)
# art_epochs_combined = art_epochs_combined.sort_values(by='start', ignore_index=True)

### Add in a buffer before and after each epoch if desired.

In [ ]:
def add_epoch_buffer(epoch_df: pd.DataFrame, buffer_sec: float or int or tuple or list):
    """Extend each epoch by buffer_sec before/after start/stop of each epoch"""
    if type(buffer_sec) in [int, float]:
        buffer_sec = (buffer_sec, buffer_sec)
    else:
        assert len(buffer_sec) == 2
        
    epoch_df['start'] -= buffer_sec[0]
    epoch_df['stop'] += buffer_sec[1]

In [ ]:
add_epoch_buffer(art_epochs.to_dataframe(), 0.2)
sess.recinfo.write_epochs(art_epochs, 'art')
art_epochs.to_dataframe()

### Write `dead_times.txt` file for later processing with SpyKing Circus

In [ ]:
dead_times_txt_path: Path = sess.basepath / 'dead_times.txt'
print(f'dead_times_txt_path: {dead_times_txt_path}')
SpykingCircusIO.write_epochs(dead_times_txt_path, epochs=art_epochs)

## Try to post-hoc downsample to no avail

In [ ]:

output_lfp_path = step_perform_downsample(concatenated_file_output_path=concatenated_file_output_path)

## try specific output path
# output_lfp_path = step_perform_downsample(concatenated_file_output_path=Path(r'W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_13-02-47\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat'), num_chan=192)
print(f'done with saving downsampled lfp')

In [ ]:
sampling_frequency: int = 30000 
num_chan: int = 200
resample_rate: int = 1250

# 1. Map the raw binary file (doesn't load into RAM yet)
recording = se.read_binary(
    file_paths=[concatenated_file_output_path.as_posix()], 
    sampling_frequency=sampling_frequency, 
    # channel_ids=np.arange(num_chan),
    # num_chan=num_chan, 
    dtype='int16'
)
recording

In [ ]:

# 2. Set up the downsampling node (applies anti-aliasing filter automatically)
recording_lfp = spre.resample(recording, resample_rate=resample_rate)

output_lfp_path: Path = concatenated_file_output_path.with_suffix('.lfp').resolve()
print(f'trying to write to: "{output_lfp_path.as_posix()}"')
# 3. Execute and write to a flat binary file in chunks
# n_jobs=-1 uses all CPU cores for faster processing
write_binary_recording(
    recording_lfp, 
    file_paths=output_lfp_path.as_posix(), 
    dtype='int16', 
    n_jobs=-1, 
    chunk_duration='1s'
)
print(f'\tdone.')

# Old stuff

In [ ]:
import spikeinterface.full as si

# recording = si.read_XXXX('/path/to/my/recording')
full_dat_file = Path(r"W:\Data\Bapun\RatS\Day1Openfield\RatS-Day1Openfield.dat").resolve()
assert full_dat_file.exists()

sampling_frequency=30000
num_chan=200
resample_rate=1250

# 1. Map the raw binary file (doesn't load into RAM yet)
# se.read_phy
spiking_circus_loaded = se.read_spykingcircus(r'W:\Data\Bapun\RatS\Day1Openfield\spyk-circ\RatS-Day1Openfield')
spiking_circus_loaded

In [ ]:

recording = se.read_binary(
    file_paths=full_dat_file.as_posix(), 
    sampling_frequency=sampling_frequency, 
    num_chan=num_chan, 
    dtype='int16'
)
recording

In [ ]:

recording_filtered = si.bandpass_filter(recording)
sorting = si.run_sorter('YYYYY', recording_filtered)


job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

# make the SortingAnalyzer with necessary and some optional extensions
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered,
                                              format="binary_folder", folder="/my_sorting_analyzer",
                                              **job_kwargs)
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
print("Hello from rats!")
raw_data_path: Path = Path(r'W:/Data/Bapun/RatS/Day1Openfield/Raw_data').resolve()
print(f'raw_data_path: "{raw_data_path.as_posix()}"')

## find all constitutent "continuous.dat" files recurrsively in all subdirectories: "W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_10-24-24\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat"
found_raw_data_paths = ["W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-20-27/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
                        # "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-24-24/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat", ## BAD ONE, only has 32 channels, skip
                        "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_13-02-47/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
                        "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_14-30-32/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
                        "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_15-06-02/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
]    
## *-24-24 is a bad one with only 30 good channels!

## Iterate through and make proper paths, check their existance
found_raw_data_paths: List[Path] = [Path(v).resolve() for v in found_raw_data_paths]
## could assert that they all exist... but let's NOT!

## make the "spyk-circ" output directory
spyk_circ_output_dir: Path = Path('W:/Data/Bapun/RatS/Day1Openfield/spyk-circ').resolve()
spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P

## Copy the concatenated files to the output directory
concatenated_file_output_path: Path = RawDataInitializationMixin.step_perform_concat(found_raw_data_paths=found_raw_data_paths, spyk_circ_output_dir=spyk_circ_output_dir)
print(f'have concatenated_file_output_path: "{concatenated_file_output_path.as_posix()}"')
concatenated_file_output_path


# Test for Day4

In [1]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin, windows_to_wsl_path_if_needed

## Bapun Format:
basedir: Path = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
# basedir: Path = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
        
n_channels: int = 195
# dat_file_sampling_rate: int = 30000
basename: str = 'RatS-Day4Openfield'
# excluded_data_datetimes = ['2020-11-25_10-24-24']
excluded_data_datetimes = [] # ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\utils\mixins\time_slicing.py:405: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class TimePointEventAccessor(TimeColumnAliasesProtocol, TimeSlicableObjectProtocol, DataframeMetadataProtocol):


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [ ]:

sess = RawDataInitializationMixin.run_all(basedir=basedir,
                                          basename=None, excluded_data_datetimes=None, n_channels=None, dat_file_sampling_rate=None, 
                                        #   valid_reference_session_basepath=windows_to_wsl_path_if_needed(Path("W:/Data/Bapun/RatS/Day5TwoNovel")), ref_basename = 'RatS-Day5TwoNovel-2020-12-04_07-55-09', ## reference session from which to create a new session (or copy missing probe/params/etc as needed),
                                        valid_reference_session_basepath=windows_to_wsl_path_if_needed(Path("/home/halechr/FastData/Bapun/RatS/Day1Openfield")), ref_basename = 'RatS-Day1Openfield', ## reference session from which to create a new session (or copy missing probe/params/etc as needed),
    # basename=basename, excluded_data_datetimes=excluded_data_datetimes, n_channels=n_channels, dat_file_sampling_rate=dat_file_sampling_rate,
)

#### Manual Processing

In [ ]:
# basedir = sess.basedir
# print(f'n_channels: {n_channels}')
# print(f'dat_file_sampling_rate: {dat_file_sampling_rate}')
# print(f'eeg_sampling_rate: {eeg_sampling_rate}')

raw_data_path: Path = basedir.joinpath('Raw_data')
assert raw_data_path.exists(), f"raw_data_path: '{raw_data_path.as_posix()}' does not exist!"
print(f'raw_data_path: "{raw_data_path.as_posix()}"')

## make the "spyk-circ" output directory
spyk_circ_output_dir: Path = basedir.joinpath('spyk-circ')
spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P

# ## INPUTS: raw_data_path: Path # Path(r'W:/Data/Bapun/RatS/Day1Openfield/Raw_data').resolve()
# print(f'raw_data_path: "{raw_data_path.as_posix()}"')

# ## find all constitutent "continuous.dat" files recurrsively in all subdirectories: "W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_10-24-24\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat"
# found_raw_data_paths = ["W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-20-27/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         # "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-24-24/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat", ## BAD ONE, only has 32 channels, skip
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_13-02-47/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_14-30-32/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_15-06-02/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
# ]    
# ## *-24-24 is a bad one with only 30 good channels!

# ## Iterate through and make proper paths, check their existance
# found_raw_data_paths: List[Path] = [Path(v).resolve() for v in found_raw_data_paths]
## could assert that they all exist... but let's NOT!
# excluded_data_datetimes = ['2020-11-25_10-24-24']


included_only_datetime_df, datetime_csv_out_path, found_raw_data_paths, all_datetime_df, all_found_files_dict = RawDataInitializationMixin.build_session_datetime_csv(raw_data_path, basename=basename, excluded_data_datetimes=excluded_data_datetimes,
                                                                                                                # n_channels=n_channels, sampling_rate=dat_file_sampling_rate,
                                                                                        )
included_only_datetime_df
all_datetime_df

In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin

# basedir = sess.basedir
print(f'n_channels: {n_channels}')
print(f'dat_file_sampling_rate: {dat_file_sampling_rate}')
print(f'eeg_sampling_rate: {eeg_sampling_rate}')

raw_data_path: Path = basedir.joinpath('Raw_data')
assert raw_data_path.exists(), f"raw_data_path: '{raw_data_path.as_posix()}' does not exist!"
print(f'raw_data_path: "{raw_data_path.as_posix()}"')

## make the "spyk-circ" output directory
spyk_circ_output_dir: Path = basedir.joinpath('spyk-circ')
spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P

# ## INPUTS: raw_data_path: Path # Path(r'W:/Data/Bapun/RatS/Day1Openfield/Raw_data').resolve()
# print(f'raw_data_path: "{raw_data_path.as_posix()}"')

# ## find all constitutent "continuous.dat" files recurrsively in all subdirectories: "W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_10-24-24\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat"
# found_raw_data_paths = ["W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-20-27/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         # "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-24-24/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat", ## BAD ONE, only has 32 channels, skip
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_13-02-47/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_14-30-32/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
#                         "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_15-06-02/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
# ]    
# ## *-24-24 is a bad one with only 30 good channels!

# ## Iterate through and make proper paths, check their existance
# found_raw_data_paths: List[Path] = [Path(v).resolve() for v in found_raw_data_paths]
## could assert that they all exist... but let's NOT!
# excluded_data_datetimes = ['2020-11-25_10-24-24']


datetime_df, datetime_csv_out_path, found_raw_data_paths, all_datetime_df = RawDataInitializationMixin.build_session_datetime_csv(raw_data_path, basename=basename, excluded_data_datetimes=excluded_data_datetimes,
                                                                                                                n_channels=n_channels, sampling_rate=dat_file_sampling_rate,
                                                                                        )
datetime_df
all_datetime_df

In [ ]:
process_resample -f 30000,1250 -n 200 '/home/halechr/FastData/Bapun/RatS/Day4Openfield/spyk-circ/RatS-Day4Openfield.dat' '/home/halechr/FastData/Bapun/RatS/Day4Openfield/RatS-Day4Openfield.eeg'

In [ ]:
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin

## INPUTS: basedir
replace_dict = RawDataInitializationMixin.step_update_session_files_n_channels(basedir=basedir, session_stem=basename, n_channels=195, dry_run=True, debug_print=True)
print(replace_dict)